In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras import layers, Model, callbacks, optimizers
from tensorflow.keras import utils
import matplotlib.pyplot as plt
import ipywidgets as widgets


2026-06-19 20:26:57.365007: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-19 20:26:57.387516: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# read data
df_data = pd.read_parquet('../../Data/training_data.parquet.gzip')
print(df_data.shape)
len(df_data.index.get_level_values(0).unique())

(1825488, 8)


336

In [3]:
# calculate daily changes by ticker
df_analysis = df_data.groupby(level='ticker', as_index=False).apply(pd.DataFrame.pct_change)

# calculate the standard deviation of close price daily change by ticker
df_analysis = df_analysis.groupby(level='ticker', as_index=True)['Close'].apply(np.std)

# sort valeus
df_analysis.sort_values(inplace=True)

# pick tickers: 2*most_volatile, 2*most_stable, 2*in_middle
mid = int(len(df_analysis.index)/2)
tickers_ft = df_analysis.index[:5].append(df_analysis.index[mid:mid+5]).append(df_analysis.index[-5:])
tickers_ft = tickers_ft.tolist()[:10]
tickers_ft

['CET', 'AJG', 'FNWD', 'CFFN', 'RY', 'FDBC', 'VABK', 'FCF', 'PFC', 'BUSE']

In [4]:
# common variables

seq_length = 7   # use the last 7 days to predict the next day
features = ["Open", "High", "Low", "Close"]
seq_length = 7
idx = pd.IndexSlice


In [5]:
def get_ticker_df(ticker, features=features):
    return df_data.loc[idx[ticker, :], features]

def create_sequence_dataset(df, seq_length=seq_length, batch_size=None, shuffle=False, seed=88):
    return utils.timeseries_dataset_from_array(
        data=df.to_numpy(dtype=np.float32),
        targets=df["Close"].iloc[seq_length:].to_numpy(dtype=np.float32),
        sequence_length=seq_length,
        batch_size=batch_size,
        shuffle=shuffle,
        seed=seed,
    )

def dataset_to_numpy(dataset):
    x_list, y_list = [], []
    for x, y in dataset:
        x_list.append(x.numpy())
        y_list.append(y.numpy())
    return np.stack(x_list), np.asarray(y_list)

## Similarity, Graph, Evaluate by dataset

In [6]:
import sys
import os
sys.path.insert(0, '.')
from CKA import linear_CKA

model_layers = [9]
model_states = ['optimal']

os.makedirs('retrain_outputs_CKA', exist_ok=True)

evaluation_mape = []

for num_layers in model_layers:
    for state in model_states:
        MODEL_PATH = f'retrain_saved_models/Retrained_{num_layers}_Layers_{state}.keras'
        model = tf.keras.models.load_model(MODEL_PATH)
        model.compile(loss="mean_absolute_percentage_error")

        # Build a model returning all representations except the final Dense output:
        # all_reprs[0] = model input, all_reprs[k] = k-th hidden LSTM output
        all_layer_outputs = [model.layers[i].output for i in range(len(model.layers) - 1)]
        repr_model = tf.keras.Model(inputs=model.input, outputs=all_layer_outputs)

        for ticker in tickers_ft:
            df_ticker = get_ticker_df(ticker)
            dataset = create_sequence_dataset(df_ticker, batch_size=None, shuffle=False)
            x_all, y_all = dataset_to_numpy(dataset)

            # all_reprs[j]: shape (n_samples, seq_len, d_j)
            all_reprs = repr_model.predict(x_all, verbose=0)

            n_samples = x_all.shape[0]
            n_hidden = len(model.layers) - 2  # exclude InputLayer and Dense

            # Per-sample intra-layer CKA: for each sample, compute CKA(layer_input, layer_output)
            # treating the seq_len timesteps as the sample dimension within each data sample
            cka_matrix = np.zeros((n_samples, n_hidden))
            for sample_idx in range(n_samples):
                for layer_idx in range(n_hidden):
                    x_in  = all_reprs[layer_idx][sample_idx]      # (seq_len, d_in)
                    x_out = all_reprs[layer_idx + 1][sample_idx]  # (seq_len, d_out)
                    cka_matrix[sample_idx, layer_idx] = linear_CKA(x_in, x_out)

            # Save CKA results as CSV
            df_cka = pd.DataFrame(
                cka_matrix,
                columns=[f"Layer_{i + 1}" for i in range(n_hidden)]
            )
            df_cka.index.name = "sample_idx"
            df_cka.to_csv(f'retrain_outputs_CKA/{num_layers}Layers_{state}_{ticker}_cka.csv')

            # Visualization: one line per samplecolumns=ticker_order, one graph per (num_layers, state, ticker)
            fig, ax = plt.subplots(figsize=(10, 6))
            layer_indices = list(range(1, n_hidden + 1))
            alpha = max(0.05, min(0.3, 30.0 / n_samples))
            for sample_idx in range(n_samples):
                ax.plot(layer_indices, cka_matrix[sample_idx],
                        alpha=alpha, linewidth=0.6, color='steelblue')
            ax.set_xlabel('Hidden Layer Index')
            ax.set_ylabel('CKA Similarity Score')
            ax.set_title(f'Intra-layer CKA  |  {num_layers} Layers  |  {state}  |  {ticker}')
            ax.set_xticks(layer_indices)
            ax.set_xticklabels([f'Layer {i}' for i in layer_indices])
            plt.tight_layout()
            plt.savefig(f'retrain_outputs_CKA/{num_layers}Layers_{state}_{ticker}_cka.png', dpi=150)
            plt.close()

            # Evaluation
            mape = float(model.evaluate(x_all, y_all, verbose=0))
            evaluation_mape.append({
                "num_layers": num_layers,
                "model_state": state,
                "ticker": ticker,
                "mape": mape
            })

# Save evaluation
df_evaluation = pd.DataFrame(evaluation_mape)
ticker_order = df_evaluation["ticker"].drop_duplicates().tolist()

df_eval = (
    df_evaluation.pivot_table(
        index=["num_layers", "model_state"],
        columns="ticker",
        values="mape",
    )
    .reindex(columns=ticker_order)
    .reset_index()
)
df_eval.columns.name = None
df_eval.to_csv('retrain_evaluation/evaluation_mape_by_dataset_5.csv', index=False)


2026-06-19 20:26:59.002429: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-19 20:26:59.006398: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-19 20:26:59.007366: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

## Evaulate with all dataset

In [7]:
import sys
import os
sys.path.insert(0, '.')

model_layers = [9]
model_states = ['optimal']

os.makedirs('outputs_CKA', exist_ok=True)

evaluation_mape = []
df_ticker = pd.DataFrame()
for ticker in tickers_ft:
    if df_ticker.empty:
        df_ticker = get_ticker_df(ticker)
    else:
        df_ticker = pd.concat([df_ticker, get_ticker_df(ticker)], axis=0)


for num_layers in model_layers:
    for state in model_states:
        MODEL_PATH = f'retrain_saved_models/Retrained_{num_layers}_Layers_{state}.keras'
        model = tf.keras.models.load_model(MODEL_PATH)
        model.compile(loss="mean_absolute_percentage_error")

        # Build a model returning all representations except the final Dense output:
        # all_reprs[0] = model input, all_reprs[k] = k-th hidden LSTM output
        all_layer_outputs = [model.layers[i].output for i in range(len(model.layers) - 1)]
        repr_model = tf.keras.Model(inputs=model.input, outputs=all_layer_outputs)

        dataset = create_sequence_dataset(df_ticker, batch_size=None, shuffle=False)
        x_all, y_all = dataset_to_numpy(dataset)

        # Evaluation
        mape = float(model.evaluate(x_all, y_all, verbose=0))
        evaluation_mape.append({
            "num_layers": num_layers,
            "model_state": state,
            "mape": mape
        })
        
df_evaluation = pd.DataFrame(evaluation_mape)
df_eval = (
    df_evaluation.pivot_table(
        index="model_state",
        columns="num_layers",
        values="mape"
    )
    .reindex()
    .reset_index()
)
df_eval.columns.name = None
df_eval.to_csv('retrain_evaluation/evaluation_mape_all_9.csv', index=False)

2026-06-19 20:28:18.656863: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
